# Cell 1 — Optional install fallback

In [1]:
from importlib.metadata import version, PackageNotFoundError
import subprocess, sys

REQS = {
    "segmentation-models-pytorch": "0.3.3",
    "timm": "0.9.2",
}

def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

current = {pkg: pkg_version(pkg) for pkg in REQS}
need_install = any(current[pkg] != wanted for pkg, wanted in REQS.items())

if need_install:
    print("Installing compatible SMP/TIMM package set...")
    print("Current versions:", current)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "segmentation-models-pytorch==0.3.3",
        "timm==0.9.2",
    ])
else:
    print("Compatible SMP/TIMM already available:", current)

import segmentation_models_pytorch as smp
import timm

print("Ready:", {
    "segmentation-models-pytorch": version("segmentation-models-pytorch"),
    "timm": version("timm"),
})

print("Note: gradio is intentionally NOT imported in Kaggle. This notebook only needs it in the exported demo package.")

Installing compatible SMP/TIMM package set...
Current versions: {'segmentation-models-pytorch': None, 'timm': '1.0.25'}
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.0 MB/s eta 0:00:00
Ready: {'segmentation-models-pytorch': '0.3.3', 'timm': '0.9.2'}
Note: gradio is intentionally NOT imported in Kaggle. This notebook only needs it in the exported demo package.


# Cell 2 — Imports, config, folders

In [2]:
import os
import gc
import cv2
import json
import math
import shutil
import zipfile
import random
import warnings
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
IMAGE_SIZE = 448
BATCH_SIZE = 8
NUM_WORKERS = 2

# Phase-13 packaging-side cleanup only
# Set these to 0 / False if you want raw thresholded masks with no cleanup.
APPLY_PACKAGING_CLOSE = True
PACKAGING_CLOSE_KERNEL = 3
PACKAGING_MIN_COMPONENT_AREA = 20

SAVE_ALL_BINARY_MASKS = True
MAX_MONTAGE_IMAGES = 24

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/output/phase13")
MASK_DIR = OUT_DIR / "phase13_val_masks"
OVERLAY_DIR = OUT_DIR / "phase13_selected_overlays"
DEMO_DIR = OUT_DIR / "phase13_demo_package"
EXAMPLES_DIR = DEMO_DIR / "examples"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)
DEMO_DIR.mkdir(parents=True, exist_ok=True)
EXAMPLES_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)

DEVICE: cuda
OUT_DIR: /kaggle/working/output/phase13


# Cell 3 — Auto-discover Kaggle inputs

In [3]:
def find_file(filename: str) -> Path:
    matches = sorted(ROOT.rglob(filename))
    assert matches, f"Could not find file: {filename}"
    return matches[0]

def find_dir_named(dirname: str, required_children=None) -> Path:
    required_children = required_children or []
    matches = []
    for p in ROOT.rglob(dirname):
        if p.is_dir() and all((p / child).exists() for child in required_children):
            matches.append(p)
    assert matches, f"Could not find directory: {dirname}"
    return matches[0]

# crack segmentation dataset
CRACK_ROOT = find_dir_named("crack_segmentation_dataset", required_children=["images", "masks"])
CRACK_IMAGES_DIR = CRACK_ROOT / "images"
CRACK_MASKS_DIR = CRACK_ROOT / "masks"

# phase files
VAL_SPLIT = find_file("val_split.csv")
TRAIN_SPLIT = find_file("train_split.csv")
PHASE11B_FINALIST = find_file("phase11b_finalist_selection.json")
PHASE12_TEMP = find_file("phase12_temperature_fit.json")
PHASE12_OP = find_file("phase12_final_operating_point.json")
PHASE11_ROOT = find_dir_named("Phase-11")

# optional protocol
phase6_matches = sorted(ROOT.rglob("phase6_protocol.md"))
PHASE6_PROTOCOL = phase6_matches[0] if phase6_matches else None

print("CRACK_ROOT:", CRACK_ROOT)
print("VAL_SPLIT:", VAL_SPLIT)
print("TRAIN_SPLIT:", TRAIN_SPLIT)
print("PHASE11B_FINALIST:", PHASE11B_FINALIST)
print("PHASE12_TEMP:", PHASE12_TEMP)
print("PHASE12_OP:", PHASE12_OP)
print("PHASE11_ROOT:", PHASE11_ROOT)
print("PHASE6_PROTOCOL:", PHASE6_PROTOCOL)

CRACK_ROOT: /kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset
VAL_SPLIT: /kaggle/input/datasets/arman00p/phase-3-csv-files/val_split.csv
TRAIN_SPLIT: /kaggle/input/datasets/arman00p/phase-3-csv-files/train_split.csv
PHASE11B_FINALIST: /kaggle/input/datasets/arman00p/phase11b-finalist-selection-json/phase11b_finalist_selection.json
PHASE12_TEMP: /kaggle/input/datasets/arman00p/phase12-calibration-reliability-finalistt/Phase12_Calibration_Reliability_Finalist/phase12_temperature_fit.json
PHASE12_OP: /kaggle/input/datasets/arman00p/phase12-calibration-reliability-finalistt/Phase12_Calibration_Reliability_Finalist/phase12_final_operating_point.json
PHASE11_ROOT: /kaggle/input/datasets/arman00p/phase11-cracksafe-mrc-training-ablation/Phase-11
PHASE6_PROTOCOL: /kaggle/input/datasets/arman00p/phase6-protocol-md/phase6_protocol.md


# Cell 4 — Read control JSONs and resolve validation image paths

In [4]:
with open(PHASE11B_FINALIST, "r") as f:
    finalist_info = json.load(f)

with open(PHASE12_TEMP, "r") as f:
    phase12_temp = json.load(f)

with open(PHASE12_OP, "r") as f:
    phase12_op = json.load(f)

val_df = pd.read_csv(VAL_SPLIT).copy()

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

img_col = pick_col(val_df, ["image_path", "img_path", "image", "filename", "file_name", "name", "id"])
mask_col = pick_col(val_df, ["mask_path", "mask", "mask_name"])
has_crack_col = pick_col(val_df, ["has_crack", "label", "target"])
source_col = pick_col(val_df, ["source", "dataset_source", "domain"])

assert has_crack_col is not None, "Could not find has_crack column in val_split.csv"
assert img_col is not None or mask_col is not None, "Could not find image/mask filename column"

name_source_col = img_col if img_col is not None else mask_col

val_df["filename"] = val_df[name_source_col].astype(str).map(lambda x: Path(x).name)
val_df["has_crack"] = val_df[has_crack_col].astype(int)
val_df["source"] = val_df[source_col].astype(str) if source_col is not None else "unknown"

val_df["image_path_resolved"] = val_df["filename"].map(lambda x: str(CRACK_IMAGES_DIR / x))
val_df["mask_path_resolved"] = val_df["filename"].map(lambda x: str(CRACK_MASKS_DIR / x))

assert val_df["image_path_resolved"].map(lambda p: Path(p).exists()).all(), "Some validation images were not found"
assert val_df["mask_path_resolved"].map(lambda p: Path(p).exists()).all(), "Some validation masks were not found"

print("Selected ablation:", finalist_info["selected_ablation"])
print("Phase-12 temperature:", phase12_temp["temperature"])
print("Phase-12 recommended mask threshold:", phase12_op["recommended_mask_threshold"])
print("Validation size:", len(val_df))

val_df[["filename", "has_crack", "source", "image_path_resolved"]].head()

Selected ablation: Boundary + clDice + Lovasz
Phase-12 temperature: 1.5564953088760376
Phase-12 recommended mask threshold: 0.55
Validation size: 1695


,filename,has_crack,source,image_path_resolved
0,Volker_DSC01686_242_386_1564_1468.jpg,1,volker,/kaggle/input/datasets/lakshaymiddha/crack-seg...
1,GAPS384_train_1166_541_641.jpg,1,gaps384,/kaggle/input/datasets/lakshaymiddha/crack-seg...
2,cracktree200_6243.jpg,1,cracktree200,/kaggle/input/datasets/lakshaymiddha/crack-seg...
3,Rissbilder_for_Florian_9S6A3121_813_2194_2441_...,1,rissbilder,/kaggle/input/datasets/lakshaymiddha/crack-seg...
4,noncrack_noncrack_concrete_wall_23_9.jpg.jpg,0,noncrack,/kaggle/input/datasets/lakshaymiddha/crack-seg...


# Cell 5 — Resolve selected checkpoint

In [5]:
def normalize_name(s: str) -> str:
    return "".join(ch.lower() for ch in str(s) if ch.isalnum())

checkpoint_from_phase12 = Path(phase12_op["checkpoint_path"])
if checkpoint_from_phase12.exists():
    BEST_CKPT = checkpoint_from_phase12
else:
    label_to_folder_key = {
        "Baseline": "ablationallfalse",
        "Boundary": "ablationonetrue",
        "Boundary + clDice": "ablationalltwotrue",
        "Boundary + clDice + Lovasz": "ablationallthreetrue",
        "Boundary + clDice + Lovasz + FDA + MRC": "ablationalltrue",
    }

    selected_ablation = finalist_info["selected_ablation"]
    folder_key = label_to_folder_key[selected_ablation]

    selected_run_dir = None
    for p in sorted(PHASE11_ROOT.iterdir()):
        if p.is_dir() and folder_key in normalize_name(p.name):
            selected_run_dir = p
            break

    assert selected_run_dir is not None, f"Could not resolve run dir for {selected_ablation}"
    best_ckpts = sorted(selected_run_dir.glob("*_best.pt"))
    assert best_ckpts, f"No *_best.pt found in {selected_run_dir}"
    BEST_CKPT = best_ckpts[0]

TEMPERATURE = float(phase12_temp["temperature"])
MASK_THRESHOLD = float(phase12_op["recommended_mask_threshold"])
ENCODER_NAME = str(phase12_op["encoder_name"])

print("BEST_CKPT:", BEST_CKPT)
print("ENCODER_NAME:", ENCODER_NAME)
print("TEMPERATURE:", TEMPERATURE)
print("MASK_THRESHOLD:", MASK_THRESHOLD)

BEST_CKPT: /kaggle/input/datasets/arman00p/phase11-cracksafe-mrc-training-ablation/Phase-11/phase11-cracksafe-mrc-training_ablationAll_ThreeTrue/phase11_full_cracksafe_mrc_unetpp_effb3_448_best.pt
ENCODER_NAME: efficientnet-b3
TEMPERATURE: 1.5564953088760376
MASK_THRESHOLD: 0.55


# Cell 6 — Build and load finalist model

In [6]:
def extract_state_dict(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                state_dict = ckpt_obj[key]
                break
        else:
            if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
                state_dict = ckpt_obj
            else:
                raise ValueError("Could not locate state dict inside checkpoint")
    else:
        raise ValueError("Unsupported checkpoint format")

    cleaned = {}
    for k, v in state_dict.items():
        nk = k
        for prefix in ["module.", "model."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v
    return cleaned

def build_unetpp_model(encoder_name):
    return smp.UnetPlusPlus(
        encoder_name=encoder_name,
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
    )

ckpt_obj = torch.load(str(BEST_CKPT), map_location="cpu")
state_dict = extract_state_dict(ckpt_obj)

encoder_candidates = [ENCODER_NAME, "efficientnet-b3", "timm-efficientnet-b3"]
encoder_candidates = list(dict.fromkeys(encoder_candidates))

best_model = None
best_encoder = None
best_preprocess_fn = None
best_score = None

for enc in encoder_candidates:
    try:
        model_try = build_unetpp_model(enc)
        incompat = model_try.load_state_dict(state_dict, strict=False)
        score = len(incompat.missing_keys) + len(incompat.unexpected_keys)
        print(f"Encoder={enc} | missing={len(incompat.missing_keys)} | unexpected={len(incompat.unexpected_keys)} | score={score}")
        if best_score is None or score < best_score:
            best_score = score
            best_model = model_try
            best_encoder = enc
            best_preprocess_fn = smp.encoders.get_preprocessing_fn(enc, "imagenet")
    except Exception as e:
        print(f"Encoder={enc} failed -> {e}")

assert best_model is not None, "Failed to build/load model"

model = best_model.to(DEVICE).eval()

print("\nChosen encoder:", best_encoder)
print("Load score:", best_score)

Encoder=efficientnet-b3 | missing=0 | unexpected=0 | score=0
Encoder=timm-efficientnet-b3 | missing=494 | unexpected=572 | score=1066

Chosen encoder: efficientnet-b3
Load score: 0


# Cell 7 — Helper functions for prediction, detections, and overlays

In [7]:
def read_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def preprocess_rgb_for_model(rgb_img, preprocess_fn, image_size=448):
    resized = cv2.resize(rgb_img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
    x = preprocess_fn(resized.astype(np.float32))
    x = np.transpose(x, (2, 0, 1)).astype(np.float32)
    return x

@torch.no_grad()
def predict_prob_map_rgb(rgb_img, temperature, preprocess_fn, image_size=448):
    orig_h, orig_w = rgb_img.shape[:2]
    x = preprocess_rgb_for_model(rgb_img, preprocess_fn, image_size=image_size)
    x = torch.from_numpy(x).unsqueeze(0).to(DEVICE)

    amp_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
    with amp_ctx:
        logits = model(x).float()[0, 0]

    prob = torch.sigmoid(logits / temperature).cpu().numpy().astype(np.float32)
    prob = cv2.resize(prob, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    return prob

def clean_binary_mask(bin_mask, do_close=True, close_kernel=3, min_component_area=20):
    out = bin_mask.astype(np.uint8).copy()

    if do_close and close_kernel > 1:
        kernel = np.ones((close_kernel, close_kernel), np.uint8)
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, kernel)

    if min_component_area > 0:
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(out, connectivity=8)
        cleaned = np.zeros_like(out)
        for lab in range(1, num_labels):
            area = int(stats[lab, cv2.CC_STAT_AREA])
            if area >= min_component_area:
                cleaned[labels == lab] = 1
        out = cleaned

    return out

def mask_to_components(prob_map, bin_mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask.astype(np.uint8), connectivity=8)
    components = []

    for lab in range(1, num_labels):
        comp_mask = (labels == lab).astype(np.uint8)
        ys, xs = np.where(comp_mask > 0)
        if len(xs) == 0:
            continue

        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        area_px = int(comp_mask.sum())

        scores = prob_map[comp_mask > 0]
        score_mean = float(scores.mean()) if scores.size > 0 else 0.0
        score_p95 = float(np.percentile(scores, 95)) if scores.size > 0 else 0.0

        pts = np.column_stack([xs, ys]).astype(np.float32)
        rect = cv2.minAreaRect(pts)
        (cx, cy), (w, h), angle = rect
        box = cv2.boxPoints(rect).astype(np.int32).tolist()

        components.append({
            "area_px": area_px,
            "score_mean": score_mean,
            "score_p95": score_p95,
            "aabb": {
                "x1": x1, "y1": y1,
                "x2": x2, "y2": y2,
                "width": int(x2 - x1 + 1),
                "height": int(y2 - y1 + 1),
            },
            "obb": {
                "center_x": float(cx),
                "center_y": float(cy),
                "width": float(w),
                "height": float(h),
                "angle_deg": float(angle),
                "points": box,
            }
        })

    components = sorted(components, key=lambda d: d["area_px"], reverse=True)
    return components

def colorize_binary_mask(bin_mask):
    mask_u8 = (bin_mask.astype(np.uint8) * 255)
    rgb = np.stack([mask_u8, mask_u8, mask_u8], axis=-1)
    return rgb

def draw_overlay(rgb_img, bin_mask, components, alpha=0.35):
    overlay = rgb_img.copy()

    green = np.zeros_like(rgb_img)
    green[..., 1] = 255
    overlay = np.where(bin_mask[..., None] > 0, (1 - alpha) * overlay + alpha * green, overlay).astype(np.uint8)

    for i, comp in enumerate(components, start=1):
        a = comp["aabb"]
        cv2.rectangle(overlay, (a["x1"], a["y1"]), (a["x2"], a["y2"]), (255, 0, 0), 2)

        obb_pts = np.array(comp["obb"]["points"], dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(overlay, [obb_pts], isClosed=True, color=(255, 255, 0), thickness=2)

        label = f"{i}:{comp['score_p95']:.2f}"
        cv2.putText(
            overlay, label, (a["x1"], max(15, a["y1"] - 4)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA
        )

    return overlay

def summarize_prediction(filename, has_crack, source, prob_map, bin_mask, components):
    crack_area_px = int(bin_mask.sum())
    total_px = int(bin_mask.size)
    crack_area_ratio = float(crack_area_px / max(total_px, 1))
    pred_any = bool(crack_area_px > 0)

    if crack_area_px > 0:
        pred_scores = prob_map[bin_mask > 0]
        confidence_mean_pred = float(pred_scores.mean())
        confidence_p95_pred = float(np.percentile(pred_scores, 95))
    else:
        confidence_mean_pred = 0.0
        confidence_p95_pred = 0.0

    return {
        "filename": filename,
        "source": source,
        "has_crack": int(has_crack),
        "pred_any": int(pred_any),
        "crack_area_px": crack_area_px,
        "crack_area_ratio": crack_area_ratio,
        "num_components": int(len(components)),
        "confidence_mean_pred": confidence_mean_pred,
        "confidence_p95_pred": confidence_p95_pred,
        "components": components,
    }

# Cell 8 — Full validation inference and JSONL export

In [8]:
records = []

jsonl_path = OUT_DIR / "phase13_val_detections_from_masks.jsonl"

with open(jsonl_path, "w") as jf:
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="Phase-13 val inference"):
        filename = row["filename"]
        has_crack = int(row["has_crack"])
        source = row["source"]
        image_path = row["image_path_resolved"]

        rgb = read_rgb(image_path)
        prob_map = predict_prob_map_rgb(
            rgb,
            temperature=TEMPERATURE,
            preprocess_fn=best_preprocess_fn,
            image_size=IMAGE_SIZE
        )

        raw_bin = (prob_map >= MASK_THRESHOLD).astype(np.uint8)
        final_bin = clean_binary_mask(
            raw_bin,
            do_close=APPLY_PACKAGING_CLOSE,
            close_kernel=PACKAGING_CLOSE_KERNEL,
            min_component_area=PACKAGING_MIN_COMPONENT_AREA
        )

        components = mask_to_components(prob_map, final_bin)
        rec = summarize_prediction(filename, has_crack, source, prob_map, final_bin, components)
        records.append(rec)

        jf.write(json.dumps(rec) + "\n")

        if SAVE_ALL_BINARY_MASKS:
            mask_png = (final_bin * 255).astype(np.uint8)
            cv2.imwrite(str(MASK_DIR / f"{Path(filename).stem}_mask.png"), mask_png)

records_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "components"} for r in records
])

records_df.to_csv(OUT_DIR / "phase13_val_detection_summary.csv", index=False)

print("Saved JSONL:", jsonl_path)
print("records_df shape:", records_df.shape)
records_df.head()

Phase-13 val inference:   0%|          | 0/1695 [00:00<?, ?it/s]

Saved JSONL: /kaggle/working/output/phase13/phase13_val_detections_from_masks.jsonl
records_df shape: (1695, 9)


,filename,source,has_crack,pred_any,crack_area_px,crack_area_ratio,num_components,confidence_mean_pred,confidence_p95_pred
0,Volker_DSC01686_242_386_1564_1468.jpg,volker,1,1,8620,0.042949,1,0.814637,0.985498
1,GAPS384_train_1166_541_641.jpg,gaps384,1,1,2544,0.012675,1,0.870284,0.984565
2,cracktree200_6243.jpg,cracktree200,1,1,2258,0.011250,45,0.752137,0.902708
3,Rissbilder_for_Florian_9S6A3121_813_2194_2441_...,rissbilder,1,1,8896,0.044324,1,0.745452,0.941898
4,noncrack_noncrack_concrete_wall_23_9.jpg.jpg,noncrack,0,0,0,0.000000,0,0.000000,0.000000


# Cell 9 — Component-level summary CSV

In [9]:
component_rows = []

for rec in records:
    for j, comp in enumerate(rec["components"], start=1):
        component_rows.append({
            "filename": rec["filename"],
            "source": rec["source"],
            "has_crack": rec["has_crack"],
            "component_rank": j,
            "area_px": comp["area_px"],
            "score_mean": comp["score_mean"],
            "score_p95": comp["score_p95"],
            "aabb_x1": comp["aabb"]["x1"],
            "aabb_y1": comp["aabb"]["y1"],
            "aabb_x2": comp["aabb"]["x2"],
            "aabb_y2": comp["aabb"]["y2"],
            "obb_center_x": comp["obb"]["center_x"],
            "obb_center_y": comp["obb"]["center_y"],
            "obb_width": comp["obb"]["width"],
            "obb_height": comp["obb"]["height"],
            "obb_angle_deg": comp["obb"]["angle_deg"],
        })

component_df = pd.DataFrame(component_rows)
component_df.to_csv(OUT_DIR / "phase13_component_summary.csv", index=False)

summary_report = {
    "selected_ablation": finalist_info["selected_ablation"],
    "checkpoint_path": str(BEST_CKPT),
    "encoder_name": best_encoder,
    "temperature": TEMPERATURE,
    "mask_threshold": MASK_THRESHOLD,
    "apply_packaging_close": APPLY_PACKAGING_CLOSE,
    "packaging_close_kernel": PACKAGING_CLOSE_KERNEL,
    "packaging_min_component_area": PACKAGING_MIN_COMPONENT_AREA,
    "num_val_images": int(len(records_df)),
    "num_predicted_positive_images": int(records_df["pred_any"].sum()),
    "mean_components_per_positive_image": float(records_df.loc[records_df["pred_any"] == 1, "num_components"].mean()) if (records_df["pred_any"] == 1).any() else 0.0,
    "mean_crack_area_ratio": float(records_df["crack_area_ratio"].mean()),
}

with open(OUT_DIR / "phase13_detection_report.json", "w") as f:
    json.dump(summary_report, f, indent=2)

print(json.dumps(summary_report, indent=2))

{
  "selected_ablation": "Boundary + clDice + Lovasz",
  "checkpoint_path": "/kaggle/input/datasets/arman00p/phase11-cracksafe-mrc-training-ablation/Phase-11/phase11-cracksafe-mrc-training_ablationAll_ThreeTrue/phase11_full_cracksafe_mrc_unetpp_effb3_448_best.pt",
  "encoder_name": "efficientnet-b3",
  "temperature": 1.5564953088760376,
  "mask_threshold": 0.55,
  "apply_packaging_close": true,
  "packaging_close_kernel": 3,
  "packaging_min_component_area": 20,
  "num_val_images": 1695,
  "num_predicted_positive_images": 1482,
  "mean_components_per_positive_image": 2.4406207827260458,
  "mean_crack_area_ratio": 0.03620269336301999
}


# Cell 10 — Curated overlay set and montage

In [10]:
# Curate informative examples
cracked_good = records_df[(records_df["has_crack"] == 1) & (records_df["pred_any"] == 1)].sort_values(
    ["crack_area_ratio", "confidence_p95_pred"], ascending=[False, False]
).head(10)

no_crack_fp = records_df[(records_df["has_crack"] == 0) & (records_df["pred_any"] == 1)].sort_values(
    ["crack_area_ratio", "confidence_p95_pred"], ascending=[False, False]
).head(6)

cracked_missed = records_df[(records_df["has_crack"] == 1) & (records_df["pred_any"] == 0)].head(4)

montage_pick = pd.concat([cracked_good, no_crack_fp, cracked_missed], axis=0).drop_duplicates("filename")
montage_pick = montage_pick.head(MAX_MONTAGE_IMAGES).reset_index(drop=True)

overlay_paths = []

for _, row in tqdm(montage_pick.iterrows(), total=len(montage_pick), desc="Create selected overlays"):
    filename = row["filename"]
    image_path = val_df.loc[val_df["filename"] == filename, "image_path_resolved"].iloc[0]

    rgb = read_rgb(image_path)
    rec = next(r for r in records if r["filename"] == filename)

    # recreate mask from saved PNG if available
    mask_path = MASK_DIR / f"{Path(filename).stem}_mask.png"
    if mask_path.exists():
        final_bin = (cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    else:
        prob_map = predict_prob_map_rgb(rgb, TEMPERATURE, best_preprocess_fn, IMAGE_SIZE)
        raw_bin = (prob_map >= MASK_THRESHOLD).astype(np.uint8)
        final_bin = clean_binary_mask(
            raw_bin,
            do_close=APPLY_PACKAGING_CLOSE,
            close_kernel=PACKAGING_CLOSE_KERNEL,
            min_component_area=PACKAGING_MIN_COMPONENT_AREA
        )

    overlay = draw_overlay(rgb, final_bin, rec["components"])

    title = f"{filename} | gt={rec['has_crack']} pred={rec['pred_any']} area={rec['crack_area_ratio']:.4f}"
    canvas = overlay.copy()
    cv2.putText(
        canvas, title, (10, 20),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA
    )

    out_path = OVERLAY_DIR / f"{Path(filename).stem}_overlay.png"
    cv2.imwrite(str(out_path), cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))
    overlay_paths.append(out_path)

# build montage
if overlay_paths:
    thumbs = []
    for p in overlay_paths:
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (320, 320), interpolation=cv2.INTER_AREA)
        thumbs.append(img)

    cols = 4
    rows = math.ceil(len(thumbs) / cols)
    canvas = np.zeros((rows * 320, cols * 320, 3), dtype=np.uint8)

    for i, img in enumerate(thumbs):
        r = i // cols
        c = i % cols
        canvas[r*320:(r+1)*320, c*320:(c+1)*320] = img

    plt.figure(figsize=(16, 4 * rows))
    plt.imshow(canvas)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "phase13_detection_montage.png", dpi=220, bbox_inches="tight")
    plt.close()

print("Saved selected overlays to:", OVERLAY_DIR)
print("Saved montage to:", OUT_DIR / "phase13_detection_montage.png")

Create selected overlays:   0%|          | 0/15 [00:00<?, ?it/s]

Saved selected overlays to: /kaggle/working/output/phase13/phase13_selected_overlays
Saved montage to: /kaggle/working/output/phase13/phase13_detection_montage.png


# Cell 11 — Create demo config and copy checkpoint

In [11]:
demo_config = {
    "selected_ablation": finalist_info["selected_ablation"],
    "encoder_name": best_encoder,
    "checkpoint_path": "checkpoint.pt",
    "temperature": TEMPERATURE,
    "mask_threshold": MASK_THRESHOLD,
    "image_size": IMAGE_SIZE,
    "apply_packaging_close": APPLY_PACKAGING_CLOSE,
    "packaging_close_kernel": PACKAGING_CLOSE_KERNEL,
    "packaging_min_component_area": PACKAGING_MIN_COMPONENT_AREA,
    "note": "Prototype public demo package generated from Phase-13 Kaggle notebook."
}

with open(DEMO_DIR / "phase13_demo_config.json", "w") as f:
    json.dump(demo_config, f, indent=2)

shutil.copy2(BEST_CKPT, DEMO_DIR / "checkpoint.pt")

# add a few example images
example_pick = pd.concat([
    records_df[records_df["has_crack"] == 1].head(3),
    records_df[records_df["has_crack"] == 0].head(3),
]).drop_duplicates("filename")

examples_list = []
for _, row in example_pick.iterrows():
    src = Path(val_df.loc[val_df["filename"] == row["filename"], "image_path_resolved"].iloc[0])
    dst = EXAMPLES_DIR / src.name
    shutil.copy2(src, dst)
    examples_list.append(str(dst.name))

with open(DEMO_DIR / "examples.json", "w") as f:
    json.dump(examples_list, f, indent=2)

print("Demo package base created:", DEMO_DIR)

Demo package base created: /kaggle/working/output/phase13/phase13_demo_package


# Cell 12 — Write app.py

In [12]:
app_py = r'''
import json
from pathlib import Path
from contextlib import nullcontext

import cv2
import numpy as np
import torch
import gradio as gr
import segmentation_models_pytorch as smp

ROOT = Path(__file__).resolve().parent

with open(ROOT / "phase13_demo_config.json", "r") as f:
    CFG = json.load(f)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def build_model():
    model = smp.UnetPlusPlus(
        encoder_name=CFG["encoder_name"],
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
    )
    ckpt = torch.load(ROOT / CFG["checkpoint_path"], map_location="cpu")

    if isinstance(ckpt, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                state_dict = ckpt[key]
                break
        else:
            state_dict = ckpt
    else:
        state_dict = ckpt

    cleaned = {}
    for k, v in state_dict.items():
        nk = k
        for prefix in ["module.", "model."]:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]
        cleaned[nk] = v

    model.load_state_dict(cleaned, strict=False)
    model = model.to(DEVICE).eval()
    preprocess_fn = smp.encoders.get_preprocessing_fn(CFG["encoder_name"], "imagenet")
    return model, preprocess_fn

MODEL, PREPROCESS_FN = build_model()

def preprocess_rgb(rgb_img, image_size):
    resized = cv2.resize(rgb_img, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
    x = PREPROCESS_FN(resized.astype(np.float32))
    x = np.transpose(x, (2, 0, 1)).astype(np.float32)
    return x

@torch.no_grad()
def predict_prob_map(rgb_img):
    h, w = rgb_img.shape[:2]
    x = preprocess_rgb(rgb_img, CFG["image_size"])
    x = torch.from_numpy(x).unsqueeze(0).to(DEVICE)

    amp_ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
    with amp_ctx:
        logits = MODEL(x).float()[0, 0]

    prob = torch.sigmoid(logits / float(CFG["temperature"])).cpu().numpy().astype(np.float32)
    prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
    return prob

def clean_binary_mask(bin_mask):
    out = bin_mask.astype(np.uint8).copy()

    if CFG["apply_packaging_close"] and CFG["packaging_close_kernel"] > 1:
        kernel = np.ones((CFG["packaging_close_kernel"], CFG["packaging_close_kernel"]), np.uint8)
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, kernel)

    min_area = int(CFG["packaging_min_component_area"])
    if min_area > 0:
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(out, connectivity=8)
        cleaned = np.zeros_like(out)
        for lab in range(1, num_labels):
            area = int(stats[lab, cv2.CC_STAT_AREA])
            if area >= min_area:
                cleaned[labels == lab] = 1
        out = cleaned

    return out

def mask_to_components(prob_map, bin_mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bin_mask.astype(np.uint8), connectivity=8)
    components = []

    for lab in range(1, num_labels):
        comp_mask = (labels == lab).astype(np.uint8)
        ys, xs = np.where(comp_mask > 0)
        if len(xs) == 0:
            continue

        x1, y1, x2, y2 = int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())
        area_px = int(comp_mask.sum())
        scores = prob_map[comp_mask > 0]
        score_mean = float(scores.mean()) if scores.size > 0 else 0.0
        score_p95 = float(np.percentile(scores, 95)) if scores.size > 0 else 0.0

        pts = np.column_stack([xs, ys]).astype(np.float32)
        rect = cv2.minAreaRect(pts)
        (cx, cy), (w, h), angle = rect
        box = cv2.boxPoints(rect).astype(np.int32).tolist()

        components.append({
            "area_px": area_px,
            "score_mean": score_mean,
            "score_p95": score_p95,
            "aabb": {
                "x1": x1, "y1": y1,
                "x2": x2, "y2": y2,
                "width": int(x2 - x1 + 1),
                "height": int(y2 - y1 + 1)
            },
            "obb": {
                "center_x": float(cx),
                "center_y": float(cy),
                "width": float(w),
                "height": float(h),
                "angle_deg": float(angle),
                "points": box
            }
        })

    components = sorted(components, key=lambda d: d["area_px"], reverse=True)
    return components

def draw_overlay(rgb_img, bin_mask, components, alpha=0.35):
    overlay = rgb_img.copy()
    green = np.zeros_like(rgb_img)
    green[..., 1] = 255
    overlay = np.where(bin_mask[..., None] > 0, (1 - alpha) * overlay + alpha * green, overlay).astype(np.uint8)

    for i, comp in enumerate(components, start=1):
        a = comp["aabb"]
        cv2.rectangle(overlay, (a["x1"], a["y1"]), (a["x2"], a["y2"]), (255, 0, 0), 2)

        obb_pts = np.array(comp["obb"]["points"], dtype=np.int32).reshape(-1, 1, 2)
        cv2.polylines(overlay, [obb_pts], isClosed=True, color=(255, 255, 0), thickness=2)

        label = f"{i}:{comp['score_p95']:.2f}"
        cv2.putText(overlay, label, (a["x1"], max(15, a["y1"] - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return overlay

def predict(image, confidence_gate):
    if image is None:
        return None, None, {}, "Please upload an image."

    if image.ndim == 2:
        image = np.stack([image, image, image], axis=-1)

    rgb = image.astype(np.uint8)
    prob_map = predict_prob_map(rgb)
    bin_mask = (prob_map >= float(CFG["mask_threshold"])).astype(np.uint8)
    bin_mask = clean_binary_mask(bin_mask)
    components = mask_to_components(prob_map, bin_mask)

    pred_any = int(bin_mask.sum() > 0)
    max_prob = float(prob_map.max()) if prob_map.size > 0 else 0.0
    gate_pass = int(max_prob >= confidence_gate)

    overlay = draw_overlay(rgb, bin_mask, components)
    mask_rgb = np.stack([bin_mask * 255, bin_mask * 255, bin_mask * 255], axis=-1).astype(np.uint8)

    crack_area_px = int(bin_mask.sum())
    crack_area_ratio = float(crack_area_px / max(bin_mask.size, 1))

    remark = []
    if pred_any:
        remark.append("Crack-like pixels detected.")
    else:
        remark.append("No crack-like pixels detected at the current threshold.")

    if gate_pass:
        remark.append(f"Confidence gate passed at {confidence_gate:.2f}.")
    else:
        remark.append(f"Confidence gate not passed at {confidence_gate:.2f}.")

    summary_md = f"""
### CrackSAFE-MRC Prototype Output
- **Predicted crack/no-crack:** {"Crack" if pred_any else "No-crack"}
- **Mask threshold:** {CFG["mask_threshold"]:.2f}
- **Confidence gate:** {confidence_gate:.2f}
- **Max calibrated confidence:** {max_prob:.4f}
- **Crack area (pixels):** {crack_area_px}
- **Crack area ratio:** {crack_area_ratio:.6f}
- **Detected components:** {len(components)}
- **Remark:** {' '.join(remark)}
"""

    result_json = {
        "pred_any": pred_any,
        "confidence_gate_pass": gate_pass,
        "max_calibrated_confidence": max_prob,
        "crack_area_px": crack_area_px,
        "crack_area_ratio": crack_area_ratio,
        "num_components": len(components),
        "components": components
    }

    return overlay, mask_rgb, result_json, summary_md

examples = []
examples_dir = ROOT / "examples"
if examples_dir.exists():
    for p in sorted(examples_dir.glob("*.jpg")):
        examples.append([str(p), 0.90])

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Image(type="numpy", label="Upload wall/concrete image"),
        gr.Slider(minimum=0.50, maximum=0.99, value=0.90, step=0.01, label="Confidence gate")
    ],
    outputs=[
        gr.Image(type="numpy", label="Overlay"),
        gr.Image(type="numpy", label="Binary mask"),
        gr.JSON(label="Detections"),
        gr.Markdown(label="Summary")
    ],
    title="CrackSAFE-MRC Prototype Demo",
    description="Upload a wall/concrete image to get calibrated crack segmentation, AABB/OBB detections, crack area proxy, and confidence-aware output.",
    examples=examples if len(examples) > 0 else None,
    allow_flagging="never"
)

if __name__ == "__main__":
    demo.launch()
'''

with open(DEMO_DIR / "app.py", "w") as f:
    f.write(app_py)

print("Wrote:", DEMO_DIR / "app.py")

Wrote: /kaggle/working/output/phase13/phase13_demo_package/app.py


# Cell 13 — Write requirements.txt and README.md

In [13]:
requirements_txt = """torch
torchvision
opencv-python-headless
numpy
pandas
segmentation-models-pytorch==0.3.3
timm==0.9.2
gradio==4.44.1
huggingface_hub==0.24.6
"""

readme_md = f"""# CrackSAFE-MRC Prototype Demo

This folder was exported from the Phase-13 Kaggle notebook.

## Included
- `app.py`
- `checkpoint.pt`
- `phase13_demo_config.json`
- `examples/`
- `requirements.txt`

## Model
- Selected ablation: **{finalist_info["selected_ablation"]}**
- Encoder: **{best_encoder}**
- Temperature: **{TEMPERATURE:.6f}**
- Recommended mask threshold: **{MASK_THRESHOLD:.2f}**

## What the demo returns
- crack mask
- overlay on original image
- crack/no-crack decision
- crack area proxy
- calibrated confidence
- AABB / OBB detections
- short plain-language remark

## Deploy to Hugging Face Spaces
1. Create a new **Gradio Space**
2. Upload all files in this folder
3. Let Spaces build from `requirements.txt`
4. The app will run from `app.py`
"""

with open(DEMO_DIR / "requirements.txt", "w") as f:
    f.write(requirements_txt)

with open(DEMO_DIR / "README.md", "w") as f:
    f.write(readme_md)

print("Wrote requirements.txt and README.md")

Wrote requirements.txt and README.md


# Cell 14 — Zip the demo package

In [14]:
zip_path = OUT_DIR / "phase13_demo_package.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in DEMO_DIR.rglob("*"):
        z.write(p, arcname=p.relative_to(DEMO_DIR))

print("Saved demo zip:", zip_path)

Saved demo zip: /kaggle/working/output/phase13/phase13_demo_package.zip


# Cell 15 — Final output check

In [15]:
print("Saved files:")
for p in sorted(OUT_DIR.glob("*")):
    print("-", p.name)

Saved files:
- phase13_component_summary.csv
- phase13_demo_package
- phase13_demo_package.zip
- phase13_detection_montage.png
- phase13_detection_report.json
- phase13_selected_overlays
- phase13_val_detection_summary.csv
- phase13_val_detections_from_masks.jsonl
- phase13_val_masks


In [16]:
import os
import shutil

folder_to_zip = "/kaggle/working/output/phase13/phase13_val_masks"
zip_base_name = "/kaggle/working/output/phase13/phase13_val_masks"

if os.path.isdir(folder_to_zip):
    zip_path = shutil.make_archive(zip_base_name, "zip", folder_to_zip)
    print("ZIP created:", zip_path)
else:
    print("Folder not found:", folder_to_zip)

ZIP created: /kaggle/working/output/phase13/phase13_val_masks.zip
